In [1]:
from pathlib import Path
file_path = Path('./data/federalist_papers.txt')

In [2]:
import requests
r = requests.get('https://www.gutenberg.org/cache/epub/1404/pg1404.txt')

In [3]:
import spacy 
nlp = spacy.load('en_core_web_md')
nlp.add_pipe('sentencizer')

In [4]:
with open(file_path , 'wb') as f:
    f.write(r.content)

In [5]:
f = open(file_path, 'r', encoding='utf-8')

collection_paper = []
federalist_paper = []

for f_line in f:
    if f_line.startswith('*** END OF THE PROJECT GUTENBERG'):
        break
    if f_line.startswith('FEDERALIST No.'):
        federalist_paper = [] 
        collection_paper.append(federalist_paper)
    federalist_paper.append(f_line)


print(collection_paper[0])
print(len(collection_paper))
f.close()

['FEDERALIST No. 1\n', '\n', 'General Introduction\n', '\n', 'For the Independent Journal. Saturday, October 27, 1787\n', '\n', '\n', 'HAMILTON\n', '\n', 'To the People of the State of New York:\n', '\n', 'AFTER an unequivocal experience of the inefficacy of the subsisting\n', 'federal government, you are called upon to deliberate on a new\n', 'Constitution for the United States of America. The subject speaks its\n', 'own importance; comprehending in its consequences nothing less than the\n', 'existence of the UNION, the safety and welfare of the parts of which it\n', 'is composed, the fate of an empire in many respects the most interesting\n', 'in the world. It has been frequently remarked that it seems to have been\n', 'reserved to the people of this country, by their conduct and example,\n', 'to decide the important question, whether societies of men are really\n', 'capable or not of establishing good government from reflection and\n', 'choice, or whether they are forever destined t

In [6]:
import re 

collection_paper = [[line.strip() for line in paper if line.strip()] for paper in collection_paper]

author_list = ['HAMILTON', 'JAY', 'MADISON']
collection_dict = []


for paper in collection_paper:
    # title,location could be added 
    paper_dict = {'paper_no': 0, 'supposed_author': '', 'content': [], 'reference': []} 

    state = 'CONTENT'
    for line in paper:
        if line.startswith('FEDERALIST No'):
            paper_dict['paper_no'] = int(re.search(r'\d+', line).group())
            continue
        if line in author_list:
            paper_dict['supposed_author'] = line
            continue
        if line == 'PUBLIUS':
            paper_dict['content'].append(line)
            state = 'REFERENCE'
            continue
        if line.startswith('To the People of the State'):
            paper_dict['content'] = []
            continue
        if state == 'CONTENT':
            cleaned = re.sub(r'\(\d+\)', '', line)
            paper_dict['content'].append(cleaned)
        else:
            paper_dict['reference'].append(line)
    paper_dict['content'] = ' '.join(paper_dict['content'])
    collection_dict.append(paper_dict)

In [7]:
print(collection_dict[0])

{'paper_no': 1, 'supposed_author': 'HAMILTON', 'content': 'AFTER an unequivocal experience of the inefficacy of the subsisting federal government, you are called upon to deliberate on a new Constitution for the United States of America. The subject speaks its own importance; comprehending in its consequences nothing less than the existence of the UNION, the safety and welfare of the parts of which it is composed, the fate of an empire in many respects the most interesting in the world. It has been frequently remarked that it seems to have been reserved to the people of this country, by their conduct and example, to decide the important question, whether societies of men are really capable or not of establishing good government from reflection and choice, or whether they are forever destined to depend for their political constitutions on accident and force. If there be any truth in the remark, the crisis at which we are arrived may with propriety be regarded as the era in which that dec

In [8]:
accepted_essays = []
disputed_essays = []
disputed_no = [49,50,51,52,53,54,55,56,57,58, 62,63]

for paper in collection_dict: 
    if paper['paper_no'] in disputed_no:
        disputed_essays.append((paper['content'], paper['supposed_author']))
    else:
        accepted_essays.append((paper['content'], paper['supposed_author']))

print(len(accepted_essays))
print(len(disputed_essays))


73
12


In [9]:
labels = [labels for (sents,labels) in accepted_essays]

In [10]:
from sklearn.model_selection import train_test_split

train_set, pretest_set = train_test_split(accepted_essays, stratify=labels, random_state=42, test_size=0.2)

In [ ]:
def cat_proportions(data, cat):
    count = 0 
    for item in data: 
        if item[1]==cat:
            count +=1
    return float(count)/float(len(data))

categories = ['JAY', 'HAMILTON', 'MADISON']
rows = []
rows.append(["Category", "Overall", "Stratified train", "Stratified pretest"])
for cat in categories:
    rows.append([cat, f"{cat_proportions(accepted_essays, cat):.6f}", f"{cat_proportions(train_set, cat):.6f}", f"{cat_proportions(pretest_set, cat):.6f}"])
columns = zip(*rows)
column_widths = [max(len(item) for item in col) for col in columns] 
for row in rows:
    print(''.join(' {:{width}} '.format(row[i], width=column_widths[i]) for i in range(0, len(row))))


 Category  Overall   Stratified train  Stratified pretest 
 JAY       0.068493  0.068966          0.066667           
 HAMILTON  0.698630  0.706897          0.666667           
 MADISON   0.191781  0.189655          0.200000           


In [ ]:
# Preprocess sentences